# TMJDLC — Javtiful Downloader (Fresh Colab)

A clean, staged smoke-test notebook. Run cells from top to bottom. It updates the repository without deleting your `output` folder, checks Chromium's Linux libraries separately from FFmpeg, and prints live resolver stages.

**Runtime note:** ❌Colab resets `/content` and installed packages after a factory reset/disconnect. No notebook can preserve those across a fresh runtime; setup skips what is already present in the current runtime.

Temporary signed media URLs and request headers remain in memory and are never printed.

## 1. Setup dependencies and & Chromium

This cell installs only missing Python packages, FFmpeg, Chromium, and Chromium system libraries. Safe to rerun.

In [ ]:
import importlib.util, shutil, subprocess, sys
from pathlib import Path

missing = []
for module, package in [('playwright','playwright>=1.50,<2'), ('yt_dlp','yt-dlp>=2025.1'), ('requests','requests>=2.32,<3')]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    print('Installing missing Python packages:', ', '.join(missing), flush=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
else:
    print('Python packages already installed; skipping.', flush=True)

if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    print('Installing FFmpeg...', flush=True)
    subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'],check=True)
else:
    print('FFmpeg/FFprobe already available; skipping.', flush=True)

# Chromium shared libraries are a separate requirement from FFmpeg.
required_libs = ['libatk-1.0.so.0','libnss3.so','libX11.so.6','libgbm.so.1']
ldconfig = subprocess.run(['bash','-lc','ldconfig -p 2>/dev/null'],capture_output=True,text=True,check=False)
missing_libs = [lib for lib in required_libs if lib not in ldconfig.stdout]
if missing_libs:
    print('Installing missing Chromium system libraries:', ', '.join(missing_libs), flush=True)
    subprocess.run([sys.executable,'-m','playwright','install-deps','chromium'],check=True)
else:
    print('Chromium system libraries already available; skipping.', flush=True)

browser_cache = Path.home()/'.cache'/'ms-playwright'
if not browser_cache.exists() or not any(browser_cache.glob('chromium-*')):
    print('Installing Playwright Chromium...', flush=True)
    subprocess.run([sys.executable,'-m','playwright','install','chromium'],check=True)
else:
    print('Playwright Chromium already cached; skipping.', flush=True)
print('Setup complete.', flush=True)

## 2. Update TMJDLC repository

First run clones the repo. Later runs fetch the latest `main` branch. Existing output files are kept outside the repository checkout.

In [ ]:
import os, pathlib, subprocess, sys
WORKDIR = pathlib.Path('/content/tmjdlc-colab')
REPO_DIR = WORKDIR/'TMJDLC'
OUTPUT_DIR = WORKDIR/'output'
SCREENSHOT_DIR = OUTPUT_DIR/'screenshots'
WORKDIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
REPO_URL = 'https://github.com/Telebotfaroff/TMJDLC.git'
BRANCH = 'main'

if not (REPO_DIR/'.git').exists():
    if REPO_DIR.exists():
        import shutil; shutil.rmtree(REPO_DIR)
    print('Cloning TMJDLC...',flush=True)
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
else:
    print('Updating existing TMJDLC checkout...',flush=True)
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','FETCH_HEAD'],check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0,str(REPO_DIR))
print('Repository ready:',subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],text=True).strip())

## 3. Configure post and quality

In [ ]:
POST_URL = input('Javtiful post URL [https://javtiful.com/video/113371/dvmm-444]: ').strip() or 'https://javtiful.com/video/113371/dvmm-444'
QUALITY = input('Preferred quality (360/480/720/1080/1440/2160) [720]: ').strip() or '720'
if QUALITY not in {'360','480','720','1080','1440','2160'}:
    raise ValueError('Unsupported quality selection')
TIMEOUT_SECONDS = 30
CAPTURE_SECONDS = 12
SCREENSHOT_DIR.mkdir(parents=True,exist_ok=True)
print('Post configured. Quality:',QUALITY+'p')

## 4. Resolve stream

Progress messages identify browser launch, navigation, player startup, and playback capture. The signed stream URL is intentionally hidden.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tmjdlc.providers.javtiful import JavtifulStreamDetector
import time

def report_stage(message):
    print('[resolver]',message,flush=True)

def resolve_job():
    detector = JavtifulStreamDetector(
        timeout_ms=TIMEOUT_SECONDS*1000,
        capture_ms=CAPTURE_SECONDS*1000,
        progress_callback=report_stage,
    )
    return detector.detect(POST_URL,preferred_quality=int(QUALITY))

print('Starting resolver in a worker thread (required by Colab asyncio)...',flush=True)
started = time.monotonic()
try:
    with ThreadPoolExecutor(max_workers=1) as executor:
        stream_result = executor.submit(resolve_job).result(timeout=TIMEOUT_SECONDS+CAPTURE_SECONDS+45)
except Exception as exc:
    print('Resolver failed after',round(time.monotonic()-started,1),'seconds:',type(exc).__name__)
    raise RuntimeError('Stream resolution failed. Check the last [resolver] stage above; do not continue to download.') from None

stream_url = stream_result.stream_url
headers = dict(stream_result.headers)
duration_seconds = stream_result.duration_seconds
print('Resolved successfully in',round(time.monotonic()-started,1),'seconds.')
print('Format:',stream_result.format,'| quality:',stream_result.quality,'| duration:',round(duration_seconds,1) if duration_seconds else 'unknown')
print('Source:',stream_result.detected_from,'| signed URL hidden')

## 5. HLS playlist preflight

Checks the selected playlist before downloading. It never prints the signed URL or segment paths.

In [ ]:
import json, re, requests
manifest = {'format':stream_result.format}
if stream_result.format == 'hls':
    try:
        response = requests.get(stream_url,headers=headers,timeout=25)
        manifest['http_status'] = response.status_code
        manifest['content_type'] = response.headers.get('content-type','').split(';',1)[0]
        manifest['response_bytes'] = len(response.content)
        response.raise_for_status()
        body = response.text.strip()
        manifest['looks_like_hls'] = body.startswith('#EXTM3U')
        manifest['is_master'] = '#EXT-X-STREAM-INF' in body
        manifest['variant_count'] = len(re.findall(r'^#EXT-X-STREAM-INF:',body,flags=re.M))
        manifest['segment_count'] = len(re.findall(r'^#EXTINF:',body,flags=re.M))
        manifest['playlist_duration_seconds'] = round(sum(float(x) for x in re.findall(r'^#EXTINF:([0-9.]+)',body,flags=re.M)),2)
        manifest['has_endlist'] = '#EXT-X-ENDLIST' in body
        if not manifest['looks_like_hls']:
            try:
                parsed_body = response.json()
                if isinstance(parsed_body,dict):
                    manifest['response_json_keys'] = sorted(parsed_body.keys())
                    safe_json = {}
                    for key,value in parsed_body.items():
                        if re.search(r'token|ticket|auth|cookie|secret|key|signature|url|src',str(key),re.I):
                            safe_json[key] = '[REDACTED]'
                        elif isinstance(value,(str,int,float,bool)) or value is None:
                            safe_json[key] = value
                        else:
                            safe_json[key] = f'<{type(value).__name__}>'
                    manifest['non_hls_json_body'] = safe_json
                else:
                    manifest['non_hls_body_type'] = type(parsed_body).__name__
            except ValueError:
                manifest['non_hls_body_preview'] = re.sub(r'https?://\\S+','[URL REDACTED]',body[:120])
    except Exception as exc:
        manifest['error_type'] = type(exc).__name__
        raise RuntimeError('Could not inspect the HLS manifest. Stop here and share the error type.') from None
print(json.dumps(manifest,indent=2))
if stream_result.format == 'hls' and not manifest.get('looks_like_hls'):
    raise RuntimeError('The selected source did not return an HLS playlist (missing #EXTM3U). Download stopped. Review the safe response diagnostics and [resolver] source metadata.')
playlist_duration = manifest.get('playlist_duration_seconds',0)
if playlist_duration and duration_seconds and playlist_duration < float(duration_seconds)*0.5:
    raise RuntimeError('Selected playlist is less than half the page duration. Likely preview/incomplete source; download stopped.')

## 6. Download with yt-dlp

Uses the repository's downloader adapter. Progress is shown without exposing source URLs or headers.

In [ ]:
from tmjdlc.downloaders import YtDlpStreamDownloader
last_percent = {'value':None}
def download_progress(data):
    status = data.get('status')
    if status == 'downloading':
        percent = data.get('_percent_str','').strip()
        if percent != last_percent['value']:
            last_percent['value'] = percent
            print('Download:',percent,'| speed:',data.get('_speed_str','').strip(),'| ETA:',data.get('_eta_str','').strip(),flush=True)
    elif status == 'finished':
        print('Download finished; merging/checking output...',flush=True)

safe_name = 'javtiful_video'
video_path = YtDlpStreamDownloader().download(
    stream_result,OUTPUT_DIR,safe_name,
    progress_hook=download_progress,
    concurrent_fragments=4,
)
print('Saved:',video_path.name,'| size MiB:',round(video_path.stat().st_size/1048576,2))

## 7. Validate downloaded video

The notebook stops if FFprobe cannot read the file or if the download is substantially shorter than the player-reported duration.

In [ ]:
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size,format_name:stream=codec_type,codec_name,width,height','-of','json',str(video_path)],capture_output=True,text=True)
if probe.returncode:
    raise RuntimeError('FFprobe validation failed: '+probe.stderr[-800:])
probe_data = json.loads(probe.stdout)
actual_duration = float(probe_data.get('format',{}).get('duration') or 0)
print(json.dumps(probe_data,indent=2))
print('Player duration:',round(float(duration_seconds),1) if duration_seconds else 'unknown','| downloaded duration:',round(actual_duration,1))
if duration_seconds and actual_duration < float(duration_seconds)*0.85:
    raise RuntimeError('Downloaded duration is below 85% of expected. Treat this as incomplete; screenshots/packaging stopped.')
if actual_duration <= 0:
    raise RuntimeError('Downloaded file has no valid duration.')
print('Video validation passed.')

## 8. Generate four screenshots

In [ ]:
from pathlib import Path
timestamps = [max(1,int(actual_duration*f)) for f in (0.15,0.35,0.60,0.85)]
generated = []
for index,seconds in enumerate(timestamps,1):
    frame = SCREENSHOT_DIR/f'frame_{index}.jpg'
    result = subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(seconds),'-i',str(video_path),'-frames:v','1','-q:v','2',str(frame)],capture_output=True,text=True)
    if result.returncode == 0 and frame.exists() and frame.stat().st_size:
        generated.append(frame)
        print('Created',frame.name,'at',seconds,'seconds')
    else:
        print('Frame failed at',seconds,'seconds:',result.stderr[-300:])
if not generated:
    raise RuntimeError('No screenshots were generated')
from IPython.display import display, Image
for frame in generated:
    display(Image(filename=str(frame)))

## 9. Package test outputs

In [ ]:
import zipfile
from google.colab import files
archive = WORKDIR/'tmjdlc-colab-output.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(video_path,arcname=video_path.name)
    for frame in generated:
        zf.write(frame,arcname='screenshots/'+frame.name)
print('ZIP size MiB:',round(archive.stat().st_size/1048576,2))
files.download(str(archive))

## Troubleshooting

- If Chromium fails to launch, rerun **cell 1** and check whether the missing system-library step completed.
- If resolution takes too long, note the last `[resolver]` message and share it. That identifies whether the delay is browser startup, navigation, or playback capture.
- If playlist duration is unexpectedly short, do not proceed with the download; share the manifest diagnostic JSON.
- If FFprobe duration is much shorter than the player duration, the file is incomplete even if yt-dlp reports completion.
- Signed URLs and browser cookies should never be pasted into logs or shared.